# Raw text→image attention inspector — SmolVLM2, every layer × every head

Captures the **pre-softmax** scores `QKᵀ/√d` for every decoder layer and head, slices out the
text→image block, and lets you look at any single `(layer, head, text token)` as a 9×9 map over
the 81 visual tokens.

```
capture   S[l,h]              [L, L]        raw, pre-softmax, post-causal-mask
slice     A[l,h] = S[text_rows][:, img_cols]
                                [L_t, L_v]
store     A                   [n_layers, H, L_t, 81]
view      A[layer, head, token].reshape(9, 9)
```

**Values are raw scores, not probabilities.** They can be negative and do not sum to 1, so the
colour scale is diverging and centred at 0, with the actual min/max printed above each plot. No
softmax is applied anywhere — that is the point of this notebook.

## The one thing to know before reading any map

The chat template puts role/system tokens **before** the image block. Those text rows are
causally blind to the image — every one of their 81 entries is the mask floor, not a real score.
They are detected by position, marked `[blind]` in the dropdown, and excluded from every MEAN.
If you select one deliberately the plot says so rather than drawing noise.

## Cost

One forward pass. About 2 minutes including the model download, then everything is interactive
off the cached tensor.

## 1. Setup

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib ipywidgets
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS

MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
MASK_FLOOR = -1e4          # anything below this is a masked position, not a score

# ---- pick the image + question -------------------------------------------------
# Option A: a WEAR-VQA sample from Drive (if you have it mounted)
# Option B: set IMG_PATH / QUESTION yourself
# Option C: leave both None -> the bundled demo image
IMG_PATH = None
QUESTION = None
DATA_ROOT = "/content/drive/MyDrive/wearvqa_gaze_only"

if IMG_PATH is None and os.path.isdir(DATA_ROOT):
    js = sorted(glob.glob(os.path.join(DATA_ROOT, "*", "*.json")))
    if js:
        meta = json.load(open(js[0])); cand = js[0][:-5] + ".jpg"
        if os.path.exists(cand):
            IMG_PATH, QUESTION = cand, meta["question"]
            print(f"using WEAR-VQA sample: {os.path.basename(IMG_PATH)}")

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

if IMG_PATH:
    image = S.load_image(IMG_PATH)
else:
    image = S._load_demo_image()
    QUESTION = QUESTION or "How many cats are in the image?"
QUESTION = QUESTION or "What does this image show?"
print(f"question: {QUESTION}")
print(f"image size: {image.size}")

## 2. Capture the raw scores and slice the text→image block

In [ ]:
msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": QUESTION}]}]
prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
inp = processor(text=prompt, images=[image], return_tensors="pt").to(device)

patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))   # None = ALL layers
try:
    with torch.no_grad():
        model(**inp)
finally:
    S._unpatch_eager_globals(patched)

ids = inp["input_ids"][0].cpu()
L = int(ids.shape[0])
img_id = S._find_image_token_id(model, processor)
pad_id = tokenizer.pad_token_id
pad_id = pad_id if pad_id is not None else -(10 ** 9)

img_cols  = torch.nonzero(ids == img_id).squeeze(-1)
text_rows = torch.nonzero((ids != img_id) & (ids != pad_id)).squeeze(-1)
L_v, L_t = len(img_cols), len(text_rows)
G = int(round(math.sqrt(L_v)))

# ---- keep only language-model self-attention, slice, stack ----
raw_by_layer = {}
for m in model.modules():
    r = getattr(m, "_raw_attn_scores", None)
    if r is not None and r.shape[-1] == L and r.shape[-2] == L:
        li = int(getattr(m, "layer_idx", len(raw_by_layer)))
        raw_by_layer[li] = r[0].float()                     # [H, L, L]
    for at in ("_raw_attn_scores", "_post_attn"):
        if hasattr(m, at):
            delattr(m, at)
assert raw_by_layer, "no language-model attention captured"

layers = sorted(raw_by_layer)
A = torch.stack([raw_by_layer[l][:, text_rows][:, :, img_cols] for l in layers], 0)
del raw_by_layer; gc.collect(); torch.cuda.empty_cache()
n_layers, H = A.shape[0], A.shape[1]

# ---- token labels, and which rows are causally blind ----
toks   = tokenizer.convert_ids_to_tokens(ids[text_rows].tolist())
pieces = [RS._detok_piece(t) for t in toks]
first_img = int(img_cols.min())
blind = (text_rows < first_img)                             # [L_t] bool

qmask = None
try:
    qmask = RS.question_span_mask(pieces, QUESTION).bool()
except Exception as e:
    print("question_span_mask unavailable:", e)

print(f"sequence L={L}   image tokens L_v={L_v} ({G}x{G})   text tokens L_t={L_t}")
print(f"captured A = [n_layers={n_layers}, H={H}, L_t={L_t}, L_v={L_v}]  "
      f"{A.numel()*4/1e6:.1f} MB")
print(f"image block occupies positions {first_img}..{int(img_cols.max())}")
print(f"causally-blind text rows (before the image): {int(blind.sum())}/{L_t}")
print("\nsequence layout:")
mark = "".join("I" if bool((ids == img_id)[i]) else "t" for i in range(L))
print("  " + mark)
print("  I = image token, t = text token\n")
print("text tokens (✗ = blind, Q = in question span):")
for i, p in enumerate(pieces):
    f = "✗" if bool(blind[i]) else " "
    q = "Q" if (qmask is not None and bool(qmask[i])) else " "
    print(f"  [{i:>3}] {f}{q}  {p!r}")

## 3. Interactive — pick layer, head, token

Raw scores. Diverging colour scale centred at 0; the printed range is the true min/max for that
one `(layer, head, token)`. Different heads live on very different scales, which is exactly why
the pipeline softmaxes per head **before** averaging anything.

In [ ]:
MEAN_ALL, MEAN_Q = "MEAN (all visible text)", "MEAN (question span)"

def slice_map(layer, head, choice):
    """-> (values[81] float tensor with NaN for masked, label str)"""
    M = A[layer, head]                                   # [L_t, L_v]
    M = torch.where(M < MASK_FLOOR, torch.full_like(M, float("nan")), M)
    vis = ~blind
    if choice == MEAN_ALL:
        rows = vis
    elif choice == MEAN_Q:
        rows = vis & (qmask if qmask is not None else torch.zeros_like(vis))
        if not rows.any():
            rows = vis
    else:
        i = int(choice.split("]")[0].lstrip("["))
        if bool(blind[i]):
            return None, f"token [{i}] {pieces[i]!r} is CAUSALLY BLIND to the image"
        return M[i], f"token [{i}] {pieces[i]!r}"
    return torch.nanmean(M[rows], dim=0), f"{choice}  ({int(rows.sum())} rows)"

def render(layer, head, choice, show_values=False):
    v, label = slice_map(layer, head, choice)
    if v is None:
        print(label); return
    a = v.numpy().reshape(G, G)
    finite = a[np.isfinite(a)]
    lim = float(np.nanmax(np.abs(finite))) if finite.size else 1.0
    lim = max(lim, 1e-9)                       # a flat head would make vmin==vmax
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)

    fig, ax = plt.subplots(1, 3, figsize=(16, 5.2))
    W, Ht = image.size
    ax[0].imshow(image)
    for k in range(1, G):
        ax[0].axhline(k*Ht/G, c="w", lw=.6, alpha=.7)
        ax[0].axvline(k*W/G,  c="w", lw=.6, alpha=.7)
    ax[0].set_title("image with the 9x9 token grid", fontsize=10); ax[0].axis("off")

    im = ax[1].imshow(a, cmap="RdBu_r", norm=norm)
    ax[1].set_xticks(range(G)); ax[1].set_yticks(range(G))
    ax[1].set_title(f"RAW pre-softmax scores\nlayer {layers[layer]} | head {head}", fontsize=10)
    plt.colorbar(im, ax=ax[1], fraction=.046)
    if show_values:
        for r in range(G):
            for c in range(G):
                if np.isfinite(a[r, c]):
                    ax[1].text(c, r, f"{a[r,c]:.1f}", ha="center", va="center", fontsize=5.5)

    ax[2].imshow(image)
    ax[2].imshow(np.kron(a, np.ones((Ht//G + 1, W//G + 1)))[:Ht, :W],
                 cmap="RdBu_r", norm=norm, alpha=.55)
    ax[2].set_title("overlaid", fontsize=10); ax[2].axis("off")

    plt.suptitle(f"{label}   |   range [{np.nanmin(a):+.3f}, {np.nanmax(a):+.3f}]   "
                 f"(NOT a probability - no softmax applied)", fontsize=11)
    plt.tight_layout(); plt.show()

CHOICES = [MEAN_ALL] + ([MEAN_Q] if qmask is not None else []) + [
    f"[{i}] {p!r}" + ("  ✗blind" if bool(blind[i]) else "") for i, p in enumerate(pieces)]

try:
    import ipywidgets as W_
    from IPython.display import display
    display(W_.interactive(
        render,
        layer=W_.IntSlider(min=0, max=n_layers-1, value=n_layers//2, description="layer"),
        head=W_.IntSlider(min=0, max=H-1, value=0, description="head"),
        choice=W_.Dropdown(options=CHOICES, value=MEAN_ALL, description="token",
                           layout=W_.Layout(width="480px")),
        show_values=W_.Checkbox(value=False, description="print numbers")))
except Exception as e:
    print("ipywidgets unavailable, falling back to plain variables:", e)
    LAYER, HEAD, CHOICE, SHOW = n_layers//2, 0, MEAN_ALL, False
    render(LAYER, HEAD, CHOICE, SHOW)

## 4. Contact sheet — every head of one layer at once

Each panel gets **its own** symmetric scale, because raw scores are not comparable across heads.
The number under each panel is that head's max |score|.

In [ ]:
COLS = 8

def contact_sheet(layer, choice=MEAN_ALL, cols=COLS):
    rows = int(math.ceil(H / cols))
    fig, ax = plt.subplots(rows, cols, figsize=(1.7*cols, 1.9*rows))
    ax = np.atleast_2d(ax)
    for h in range(rows*cols):
        a_ = ax[h//cols, h % cols]; a_.axis("off")
        if h >= H:
            continue
        v, _ = slice_map(layer, h, choice)
        if v is None:
            continue
        m = v.numpy().reshape(G, G)
        fin = m[np.isfinite(m)]
        lim = max(float(np.nanmax(np.abs(fin))) if fin.size else 1.0, 1e-9)
        a_.imshow(m, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim))
        a_.set_title(f"h{h}\n{lim:.1f}", fontsize=6.5)
    plt.suptitle(f"layer {layers[layer]} - all {H} heads - {choice}   (raw scores, per-head scale)",
                 fontsize=11)
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as W_
    from IPython.display import display
    display(W_.interactive(
        contact_sheet,
        layer=W_.IntSlider(min=0, max=n_layers-1, value=n_layers//2, description="layer"),
        choice=W_.Dropdown(options=CHOICES, value=MEAN_ALL, description="token",
                           layout=W_.Layout(width="480px")),
        cols=W_.fixed(COLS)))
except Exception:
    contact_sheet(n_layers//2)

## 5. Layer sweep — one head across every layer

In [ ]:
def layer_sweep(head, choice=MEAN_ALL, cols=COLS):
    rows = int(math.ceil(n_layers / cols))
    fig, ax = plt.subplots(rows, cols, figsize=(1.7*cols, 1.9*rows))
    ax = np.atleast_2d(ax)
    for l in range(rows*cols):
        a_ = ax[l//cols, l % cols]; a_.axis("off")
        if l >= n_layers:
            continue
        v, _ = slice_map(l, head, choice)
        if v is None:
            continue
        m = v.numpy().reshape(G, G)
        fin = m[np.isfinite(m)]
        lim = max(float(np.nanmax(np.abs(fin))) if fin.size else 1.0, 1e-9)
        a_.imshow(m, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim))
        a_.set_title(f"L{layers[l]}\n{lim:.1f}", fontsize=6.5)
    plt.suptitle(f"head {head} - all {n_layers} layers - {choice}   (raw scores, per-layer scale)",
                 fontsize=11)
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as W_
    from IPython.display import display
    display(W_.interactive(
        layer_sweep,
        head=W_.IntSlider(min=0, max=H-1, value=0, description="head"),
        choice=W_.Dropdown(options=CHOICES, value=MEAN_ALL, description="token",
                           layout=W_.Layout(width="480px")),
        cols=W_.fixed(COLS)))
except Exception:
    layer_sweep(0)

## 6. Notes on reading these

* **Raw scores, no softmax.** Values are `QKᵀ/√d` after the causal mask is added. Negative is
  normal. Two heads are not comparable on the same colour scale, which is why every panel is
  scaled independently and the max |score| is printed.

* **Masked positions show as blank (NaN).** Anything below `MASK_FLOOR = -1e4` is a mask fill
  value from `transformers` (`torch.finfo(dtype).min`), not a real score.

* **Blind rows.** Text tokens before the image block cannot attend to it. They are excluded from
  MEAN and marked `✗blind` in the dropdown. Averaging them in is what once made `sink_scores`
  return an all-zero vector.

* **`A` is a plain tensor** — `[n_layers, H, L_t, 81]`. Use it directly for anything else:

  ```python
  A[12, 5, 3]                      # layer 12, head 5, text token 3 -> [81]
  A.mean((0, 1))                   # mean over layers and heads    -> [L_t, 81]
  torch.softmax(A[12, 5, 3], -1)   # if you want the probability version
  ```

* **`do_image_splitting=False`** is set in the loader, which is what gives 81 tokens on a 9×9
  grid. Turn it on and `L_v` jumps to 324+, `G` changes with it, and the capture memory grows
  as `L²`.